<a href="https://colab.research.google.com/github/sainiaman27/nassau-candy-route-analysis/blob/main/Factory_to_Customer_Shipping_Route_Efficiency_Analysis_for_Nassau_Candy_Distributor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load Dataset
df = pd.read_csv("/content/Nassau Candy Distributor.csv")

In [ ]:
# Inspect Initial Shape and Dtypes
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          10194 non-null  int64  
 1   Order ID        10194 non-null  object 
 2   Order Date      10194 non-null  object 
 3   Ship Date       10194 non-null  object 
 4   Ship Mode       10194 non-null  object 
 5   Customer ID     10194 non-null  int64  
 6   Country/Region  10194 non-null  object 
 7   City            10194 non-null  object 
 8   State/Province  10194 non-null  object 
 9   Postal Code     10194 non-null  object 
 10  Division        10194 non-null  object 
 11  Region          10194 non-null  object 
 12  Product ID      10194 non-null  object 
 13  Product Name    10194 non-null  object 
 14  Sales           10194 non-null  float64
 15  Units           10194 non-null  int64  
 16  Gross Profit    10194 non-null  float64
 17  Cost            10194 non-null 

In [ ]:
# Display First Few Rows
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90


In [ ]:
# Convert to DateTime From Object Dtype
df["Order Date"] = pd.to_datetime(df["Order Date"],errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"],errors="coerce")

/tmp/ipykernel_1146/2670660773.py:3: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["Ship Date"] = pd.to_datetime(df["Ship Date"],errors="coerce")


In [ ]:
# Drop any unpareable dates
df = df.dropna(subset=["Order Date","Ship Date"]).copy()

In [ ]:
# Raw lead time
df["Raw_lead_time_days"] = (df["Ship Date"] - df["Order Date"]).dt.days

# Reconstruct true operational dates by reversing the dataset
df["Order_Id_Year"] = df["Order ID"].str.extract(r"(\d{4})")[0].astype(int)
original_shipping_date = df["Ship Date"] - pd.Timedelta(days=2000)

def restore_order_year(row):
  dt = row["Order Date"]
  yr = row["Order_Id_Year"]
  if dt.month == 2 and dt.day == 29:
    return pd.Timestamp(year=yr, month=2, day=28)
  else:
    return pd.Timestamp(year=yr, month=dt.month, day=dt.day)

original_order_date = df.apply(restore_order_year, axis=1)
df["shipping_lead_time"] = (original_shipping_date - original_order_date).dt.days

# Remove any invalid or negative lead times
df = df[df["shipping_lead_time"] >= 0].copy()

# Geographic and Text Fields
text_cols = ["Country/Region", "City", "State/Province", "Region", "Ship Mode", "Division"]
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()
df["Postal Code"] = df["Postal Code"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()

In [ ]:
# Map Products to Factories & Coordinates
product_to_factory = {
    "Wonka Bar - Nutty Crunch Surprise": "Lot's O' Nuts",
    "Wonka Bar - Fudge Mallows": "Lot's O' Nuts",
    "Wonka Bar -Scrumdiddlyumptious": "Lot's O' Nuts",
    "Wonka Bar - Milk Chocolate": "Wicked Choccy's",
    "Wonka Bar - Triple Dazzle Caramel": "Wicked Choccy's",
    "Laffy Taffy": "Sugar Shack",
    "SweeTARTS": "Sugar Shack",
    "Nerds": "Sugar Shack",
    "Fun Dip": "Sugar Shack",
    "Fizzy Lifting Drinks": "Sugar Shack",
    "Everlasting Gobstopper": "Secret Factory",
    "Lickable Wallpaper": "Secret Factory",
    "Wonka Gum": "Secret Factory",
    "Hair Toffee": "The Other Factory",
    "Kazookles": "The Other Factory",
}

factory_coordinates = {
    "Lot's O' Nuts": (32.881893, -111.768036),
    "Wicked Choccy's": (32.076176, -81.088371),
    "Sugar Shack": (48.119140, -96.181150),
    "Secret Factory": (41.446333, -90.565487),
    "The Other Factory": (35.117500, -89.971107),
}

df["Factory"] = df["Product Name"].map(product_to_factory)
df["Factory_Lat"] = df["Factory"].map(lambda x: factory_coordinates[x][0])
df["Factory_Lon"] = df["Factory"].map(lambda x: factory_coordinates[x][1])



In [ ]:
# Engineer Route Identifiers
df["Route_State"] = df["Factory"] + " -> " + df["State/Province"]
df["Route_Region"] = df["Factory"] + " -> " + df["Region"]

In [ ]:
# Save cleaned dataset
df.to_csv("nassau_candy_cleaned.csv", index=False)
print("Saved cleaned dataset:", df.shape)
print("\nAverage Lead Time by Ship Mode (Days):")
print(df.groupby("Ship Mode")["shipping_lead_time"].agg(["count", "mean", "std", "min", "max"]))

Saved cleaned dataset: (2741, 26)

Average Lead Time by Ship Mode (Days):
                count        mean        std  min  max
Ship Mode                                             
First Class       444  124.968468  94.785293    1  327
Same Day          157  112.910828  94.400224    0  324
Second Class      551  129.444646  93.713238    2  328
Standard Class   1589  131.064821  95.155389    4  331
